# Pelajaran 13 - Memori Agen dengan Graf Pengetahuan Cognee


## Setup

Notebook ini menunjukkan cara membangun **asisten coding** cerdas dengan memori persisten menggunakan grafik pengetahuan [**Cognee**](https://www.cognee.ai/) dan **Microsoft Agent Framework** (MAF).

Cognee mengubah teks tidak terstruktur menjadi grafik pengetahuan terstruktur yang dapat ditanyakan dan didukung oleh vector embeddings — memberikan agen Anda memori jangka panjang yang kaya dan sadar hubungan.

### Apa yang Akan Anda Pelajari
1. **Membangun Grafik Pengetahuan**: Mengubah profil pengembang dan praktik terbaik menjadi pengetahuan yang terstruktur dan dapat ditanyakan.
2. **Mengintegrasikan Cognee dengan MAF**: Menggunakan fungsi `@tool` untuk membiarkan agen MAF menanyakan grafik pengetahuan Cognee.
3. **Percakapan yang Sadar Sesi**: Menjaga konteks di seluruh beberapa pertanyaan dalam sesi yang sama.
4. **Memori Jangka Panjang**: Mempertahankan pengetahuan penting di seluruh sesi dan mengambilnya kembali dalam percakapan baru.

### Prasyarat
- Python 3.9+
- Redis berjalan secara lokal (`docker run -d -p 6379:6379 redis`) untuk manajemen sesi
- Kunci API LLM (misalnya OpenAI) — setel `LLM_API_KEY` di `.env`
- `CACHING=true` di `.env` (diperlukan untuk sesi Cognee)
- Proyek Azure AI Foundry dengan model chat yang sudah diterapkan
- `AZURE_AI_PROJECT_ENDPOINT` dan `AZURE_AI_MODEL_DEPLOYMENT_NAME` di `.env`
- Azure CLI sudah diautentikasi (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Jenis Memori Agen

Notebook ini mengeksplorasi tiga tipe memori yang sama dari notebook utama Pelajaran 13, tetapi menggunakan Cognee sebagai backend memori jangka panjang:

| Tipe Memori | Mekanisme | Masa Hidup |
|---|---|---|
| **Kerja** | `agent.create_session()` (MAF) | Percakapan tunggal |
| **Jangka pendek** | Cache sesi Cognee (Redis) | Sesi tunggal |
| **Jangka panjang** | Graf pengetahuan Cognee + vektor | Permanen |

### Arsitektur Memori Cognee
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Siapkan Penyimpanan Cognee


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Bagian 1 — Membangun Basis Pengetahuan

Kami mengolah tiga jenis data untuk membuat basis pengetahuan yang komprehensif bagi asisten pengkodean kami:

1. **Profil Pengembang** — keahlian pribadi dan latar belakang teknis
2. **Praktik Terbaik Python** — Zen of Python dengan panduan praktis
3. **Percakapan Historis** — sesi tanya jawab masa lalu antara pengembang dan asisten AI


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## Visualisasikan Grafik Pengetahuan

Cognee dapat menampilkan visualisasi HTML interaktif dari entitas dan hubungan yang diekstraknya.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Memperkaya Memori dengan Memify

`memify()` menganalisis grafik pengetahuan dan menghasilkan aturan cerdas — mengidentifikasi pola, praktik terbaik, dan hubungan antara konsep.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## Bagian 2 — Agen MAF dengan Cognee Tools

Sekarang kita membuat agen MAF yang dapat menanyakan grafik pengetahuan Cognee melalui fungsi `@tool`. Ini memungkinkan agen memanfaatkan kekuatan penuh pencarian semantik yang sadar grafik sambil mempertahankan konteks percakapan melalui sesi.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## Memori Kerja dengan Sesi

`AgentSession` (dibuat melalui `agent.create_session()`) menyediakan memori kerja dalam sebuah sesi. Agen dapat merujuk kembali ke pesan-pesan sebelumnya sekaligus menanyakan grafik pengetahuan jangka panjang Cognee.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## Sesi Baru — Memori Jangka Panjang Tetap Ada

Memulai sesi baru menghapus memori kerja, tetapi grafik pengetahuan Cognee masih tersedia. Agen dapat mengambil pengetahuan jangka panjang yang sama dalam percakapan yang benar-benar baru.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Ringkasan

Dalam notebook ini Anda membangun asisten pengodean yang menggabungkan **memori kerja MAF** (`agent.create_session()`) dengan **graf pengetahuan jangka panjang Cognee**.

### Apa yang Telah Anda Pelajari
1. **Konstruksi graf pengetahuan**: Cognee mengolah teks tidak terstruktur dan membangun graf + memori vektor.
2. **Pengayaan graf dengan memify**: Fakta turunan dan hubungan lebih kaya di atas graf yang sudah ada.
3. **Integrasi MAF + Cognee**: Fungsi `@tool` memungkinkan agen MAF untuk melakukan query graf Cognee secara alami.
4. **Memori kerja + memori jangka panjang**: `AgentSession` (melalui `agent.create_session()`) menyediakan konteks sesi sementara Cognee menyediakan pengetahuan yang persisten.
5. **Pencarian terfilter dengan NodeSets**: Menargetkan subset tertentu dari graf pengetahuan (misal hanya prinsip-prinsip).

### Poin Penting
- **Cognee** mengubah teks mentah menjadi memori terstruktur yang sadar hubungan — lebih kuat daripada penyimpanan vektor datar.
- Fungsi **`@tool`** menjembatani agen MAF dan sistem pengetahuan eksternal dengan bersih.
- **`AgentSession`** (melalui `agent.create_session()`) menjaga konteks per percakapan terpisah dari pengetahuan jangka panjang.
- Graf pengetahuan yang sama melayani banyak sesi dan agen.

### Aplikasi Dunia Nyata
- **Co-pilot pengembang**: Review kode, analisis insiden, asisten arsitektur
- **Co-pilot untuk pelanggan**: Agen dukungan atas dokumen produk, FAQ, dan catatan CRM
- **Co-pilot ahli internal**: Asisten kebijakan, hukum, atau keamanan yang menalar atas pedoman
- **Lapisan data terpadu**: Menggabungkan data terstruktur dan tidak terstruktur menjadi satu graf yang dapat di-query

### Langkah Berikutnya
- Bereksperimen dengan kesadaran temporal dalam Cognee
- Mendefinisikan ontologi OWL untuk kualitas graf spesifik domain
- Menambahkan loop umpan balik pengguna untuk meningkatkan pengambilan sepanjang waktu
- Mengskalakan ke sistem multi-agen yang berbagi lapisan memori Cognee yang sama


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:  
Dokumen ini telah diterjemahkan menggunakan layanan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Meskipun kami berusaha untuk mencapai akurasi, harap diingat bahwa terjemahan otomatis mungkin mengandung kesalahan atau ketidakakuratan. Dokumen asli dalam bahasa aslinya harus dianggap sebagai sumber yang sahih. Untuk informasi penting, disarankan menggunakan terjemahan profesional manusia. Kami tidak bertanggung jawab atas kesalahpahaman atau penafsiran yang salah yang timbul dari penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
